In [126]:
import random_forest_oconnell
import pandas as pd
from sklearn.ensemble import  RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [168]:
# CONSTANTS
FEATURE_COLS = [
    "danceability",
    "energy",
    "tempo",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "loudness",
    "key",
    "mode"
]

REQUIRED_COLS = ["artist", "track"] + FEATURE_COLS
# Create df from big dataset and clean it so cols match and it only has one artist
SONGS = pd.read_csv("data/tracks_features.csv")
SONGS = SONGS.rename(columns={"name": "track"})
SONGS["artist"] = SONGS["artists"].apply(random_forest_oconnell.extract_first_artist)
SONGS = SONGS[REQUIRED_COLS]
SONGS["artist"] = SONGS["artist"].astype(str).str.strip()
SONGS["track"] = SONGS["track"].astype(str).str.strip()

In [10]:
# Loading user data from csv
song_df = pd.read_csv("data/tre_lastfm.csv", sep=",")
song_df.columns = song_df.columns.str.strip().str.lower()
song_df["count"] = 1
song_df = (
    song_df.groupby(["artist", "track"], as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
if len(song_df) > 200:
    song_df = song_df.iloc[:200]    # Trimming df if it's longer than 200 values
features = random_forest_oconnell.obtain_features(song_df)

Searching Spotify for IDs...
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200
Fetched features for batch 0 (40 songs)
Fetched features for batch 0 (40 songs)
Fetched features for batch 1 (40 songs)
Fetched features for batch 2 (40 songs)
Fetched features for batch 3 (40 songs)


In [63]:
song_df_with_feats = random_forest_oconnell.create_feature_df(song_df, features)

In [43]:
# Saving it in case I need to use it in the future
song_df_with_feats.to_csv("data/tre_lastfm_with_features.csv", index=False)

In [62]:
#Dropping rows with NA values
clean_df = song_df_with_feats.dropna(subset=FEATURE_COLS).copy()

In [45]:
# Creating user profile so we can see what they generally like
user_profile = (
    clean_df[FEATURE_COLS]
    .multiply(clean_df["count"], axis=0)
    .sum() / clean_df["count"].sum()
)

print(user_profile)

danceability          0.610216
energy                0.637947
tempo               123.622726
valence               0.440958
acousticness          0.211092
instrumentalness      0.084349
liveness              0.189028
speechiness           0.066116
loudness             -6.447882
key                   5.241970
mode                  0.563169
dtype: float64


In [103]:
# Now I'm going to remove songs that the user has listened to, so we don't recommend something they've heard already.
heard_songs = clean_df[["artist", "track"]].drop_duplicates().copy()

candidate_df = SONGS.merge(
    heard_songs,
    on=["artist", "track"],
    how="left",
    indicator=True
)
candidate_df = candidate_df[candidate_df["_merge"] == "left_only"].drop(columns=["_merge"])

In [182]:
# Inserting some unheard songs, so the model can better predict.
positive_df = clean_df.copy()
negative_df = candidate_df.sample(n=int(len(positive_df))).copy()
negative_df["count"] = 0

train_df = pd.concat([positive_df, negative_df], ignore_index=True)

X = train_df[FEATURE_COLS]
y = train_df["count"]

In [195]:
# Actually training the model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Model evaluation:")
mse = mean_squared_error(y_test, pred)
print("MSE:", mse)

Model evaluation:
MSE: 17.642818499999997


In [164]:
# For reference, here's the model's performance without adding false/noisy values
X_no_noise = clean_df[FEATURE_COLS]
y_no_noise = clean_df["count"]
# Actually training the model
X_train_no_noise, X_test_no_noise, y_train_no_noise, y_test_no_noise = train_test_split(
    X_no_noise, y_no_noise, test_size=0.2
)

model_with_no_noise = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model_with_no_noise.fit(X_train_no_noise, y_train_no_noise)

pred_no_noise = model_with_no_noise.predict(X_test_no_noise)

print("Model evaluation:")
mse = mean_squared_error(y_test_no_noise, pred_no_noise)
print("MSE:", mse)

Model evaluation:
MSE: 46.148971


In [165]:
# Getting recommendations
candidate_X = candidate_df[FEATURE_COLS]
candidate_df["predicted_count"] = model.predict(candidate_X)
recommendations = candidate_df.sort_values(
    "predicted_count",
    ascending=False
)

In [166]:
recommendations.head(20)

,artist,track,danceability,energy,tempo,valence,acousticness,instrumentalness,liveness,speechiness,loudness,key,mode,predicted_count
738922,Soda Stereo,Trátame Suavemente - Remasterizado 2007,0.738,0.801,122.857,0.5320,0.0438,0.057200,0.595,0.0284,-3.366,11,0,13.290
1084308,Tech N9ne,B.I.B.,0.704,0.804,78.056,0.5390,0.4860,0.000000,0.592,0.3330,-5.296,9,0,12.970
961431,Soda Stereo,Trátame Suavemente - Remasterizado 2007,0.738,0.789,122.852,0.5390,0.0401,0.064700,0.632,0.0283,-4.009,11,0,12.965
1160775,Shing02,In the Mood,0.682,0.647,86.011,0.4760,0.0264,0.001240,0.578,0.0503,-4.966,11,1,12.800
1074712,Woods,Holier Than No One,0.723,0.929,111.726,0.5770,0.9790,0.958000,0.910,0.2690,-5.292,7,0,12.760
363169,Muslimgauze,Minaret Speaker 3,0.699,0.989,114.037,0.0329,0.6650,0.686000,0.618,0.3150,-2.885,9,1,12.730
1128676,Hol!,The Living Dead,0.716,0.958,72.526,0.0682,0.0424,0.582000,0.710,0.0514,-0.008,8,1,12.705
485410,Travis Larson Band,Above Below (Live),0.714,0.772,121.253,0.5110,0.1560,0.265000,0.950,0.0372,-5.153,11,0,12.555
1132841,Starseed Dro,Bonnie Doone,0.704,0.754,138.002,0.5090,0.0913,0.000000,0.624,0.0403,-5.078,11,1,12.550
948283,Eddy I.,Capone,0.764,0.657,140.011,0.0710,0.0494,0.020800,0.847,0.0682,-2.737,8,1,12.485


In [128]:
# Demonstrating why I can't just run this on one song :(
single_song_df = SONGS.sample(n=1, random_state=42).copy()
single_song_df["count"] = 1
single_song_df

,artist,track,danceability,energy,tempo,valence,acousticness,instrumentalness,liveness,speechiness,loudness,key,mode,count
54449,Unk,Smokin' Sticky Sticky,0.623,0.736,87.988,0.422,0.0021,0.0,0.0691,0.402,-3.657,11,0,1


In [138]:
X_single_song = single_song_df[FEATURE_COLS]
y_single_song = single_song_df["count"]

single_song_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

single_song_model.fit(X_single_song, y_single_song)

candidate_df_single_song = SONGS.copy()
candidate_X_single_song = candidate_df_single_song[FEATURE_COLS]

candidate_df_single_song["predicted_count"] = single_song_model.predict(candidate_X_single_song)

recommendations = candidate_df_single_song.sort_values(
    "predicted_count",
    ascending=False
)

recommendations.head(20)

,artist,track,danceability,energy,tempo,valence,acousticness,instrumentalness,liveness,speechiness,loudness,key,mode,predicted_count
0,Rage Against The Machine,Testify,0.470,0.9780,117.906,0.5030,0.026100,0.000011,0.3560,0.0727,-5.399,7,1,1.0
802672,Tragederia,The Architect (Single Version),0.541,0.9690,104.016,0.0838,0.000971,0.703000,0.0908,0.0587,-5.297,2,1,1.0
802688,Martin O'Donnell,We're Not Going Anywhere,0.199,0.1380,66.964,0.0391,0.280000,0.877000,0.0927,0.0403,-21.405,3,1,1.0
802687,Martin O'Donnell,Fortress,0.770,0.7700,107.041,0.3510,0.088200,0.954000,0.2100,0.0663,-16.270,1,1,1.0
802686,Martin O'Donnell,Ashes,0.241,0.0850,82.756,0.0433,0.984000,0.968000,0.0986,0.0359,-19.295,5,0,1.0
802685,Martin O'Donnell,From the Vault,0.729,0.3930,102.968,0.4110,0.671000,0.905000,0.0961,0.0290,-19.096,6,1,1.0
802684,Martin O'Donnell,Epilogue,0.104,0.0767,67.554,0.0354,0.966000,0.958000,0.0930,0.0398,-18.684,8,1,1.0
802683,Martin O'Donnell,The Pillar of Autumn,0.322,0.4340,103.054,0.1280,0.288000,0.849000,0.1010,0.0717,-17.167,5,0,1.0
802682,Martin O'Donnell,The Package,0.276,0.2990,63.813,0.0352,0.663000,0.871000,0.0857,0.0497,-16.743,5,1,1.0
802681,Martin O'Donnell,New Alexandria,0.183,0.2650,80.417,0.0399,0.838000,0.864000,0.1100,0.0423,-19.500,1,1,1.0
